In [6]:
import io

import numpy as np
from async_geotiff import GeoTIFF
from async_geotiff import Tile as GeoTIFFTile
from obstore.store import S3Store
from PIL import Image
from sidecar import Sidecar

from lonboard import Map, RasterLayer
from lonboard.raster import EncodedImage, reshape_as_image

In [2]:
store = S3Store("sentinel-cogs", region="us-west-2", skip_signature=True)
cog_path = "sentinel-s2-l2a-cogs/18/T/WL/2026/1/S2B_18TWL_20260101_0_L2A/TCI.tif"
geotiff = await GeoTIFF.open(cog_path, store=store)

In [3]:
def render_rgb_tile(tile: GeoTIFFTile) -> EncodedImage:
    """Convert the array data from the GeoTIFF to an RGB PNG."""
    # Reshape from (bands, height, width) to (height, width, bands)
    masked_array = reshape_as_image(tile.array.as_masked())

    rgb = masked_array.data
    mask = masked_array.mask
    if mask.ndim == 0:
        # All pixels valid
        alpha = np.full(rgb.shape[:2], 255, dtype=np.uint8)
    elif mask.ndim == 2:
        alpha = (~mask).astype(np.uint8) * 255
    else:
        alpha = (~mask.any(axis=-1)).astype(np.uint8) * 255

    # Concatenate alpha axis to become (height, width, 4)
    rgba = np.concatenate([rgb, alpha[..., None]], axis=-1)

    img = Image.fromarray(rgba)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return EncodedImage(data=buf.getvalue(), media_type="image/png")

In [4]:
layer = RasterLayer.from_geotiff(geotiff, render_tile=render_rgb_tile)

In [7]:
sidecar = Sidecar(anchor="split-right")

In [8]:
m = Map(layer, height=800)

In [9]:
with sidecar:
    display(m)